# MCal Calibration Demo

This notebook demonstrates the usage of the MCal calibration framework.

In [ ]:
import sys
import os
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from calibrators import MCal, PlattCalibrator, TemperatureScaling
from utils.optimization import get_expectation
from utils.visualization import plot_training_curves

## Generate Synthetic Data

Create synthetic probability distributions for testing calibration methods.

In [ ]:
torch.manual_seed(1234)

d = 4  # Number of classes
N = 1000  # Number of samples

# Create clean (target) probabilities - uniform distribution
clean_probs = torch.ones(N, d)
clean_probs /= clean_probs.sum(dim=1, keepdim=True)

# Create ablated (biased) probabilities - biased towards first class
ablated_probs = clean_probs + torch.rand_like(clean_probs) + torch.eye(d)[0]
ablated_probs /= ablated_probs.sum(dim=1, keepdim=True)

print(f"Clean probs shape: {clean_probs.shape}")
print(f"Ablated probs shape: {ablated_probs.shape}")
print(f"Raw accuracy: {(clean_probs.argmax(dim=-1) == ablated_probs.argmax(dim=-1)).float().mean():.3f}")

## Test MCal Calibrator

In [ ]:
# Initialize and fit MCal
mcal = MCal(d)
mcal_stats = mcal.fit(
    ablated_probs, 
    clean_probs, 
    verbose=True, 
    kappa=1.0,
    lr=1e-3,
    early_stopping=False, 
    max_steps=1000
)

# Get calibrated outputs
mcal_output = mcal(ablated_probs)
print(f"MCal output shape: {mcal_output.shape}")

# Calculate expectations
one_hot_exp, prob_exp = get_expectation(mcal_output)
print(f"MCal one-hot expectation: {one_hot_exp}")
print(f"MCal prob expectation: {prob_exp}")

## Test Platt Calibrator

In [ ]:
# Initialize and fit Platt calibrator
platt = PlattCalibrator(d)
platt_stats = platt.fit(
    ablated_probs, 
    clean_probs, 
    lr=1e-3, 
    verbose=True, 
    max_steps=1000
)

# Get calibrated outputs
platt_output = platt(ablated_probs)
print(f"Platt output shape: {platt_output.shape}")

# Calculate expectations
one_hot_exp, prob_exp = get_expectation(platt_output)
print(f"Platt one-hot expectation: {one_hot_exp}")
print(f"Platt prob expectation: {prob_exp}")

## Test Temperature Scaling

In [ ]:
# Initialize and fit temperature scaling
temp_scaler = TemperatureScaling(d)
temp_stats = temp_scaler.fit(
    ablated_probs, 
    clean_probs, 
    verbose=True
)

# Get calibrated outputs
temp_output = temp_scaler(ablated_probs)
print(f"Temperature scaling output shape: {temp_output.shape}")
print(f"Learned temperature: {temp_scaler.temperature.item():.4f}")

# Calculate expectations
one_hot_exp, prob_exp = get_expectation(temp_output)
print(f"Temperature one-hot expectation: {one_hot_exp}")
print(f"Temperature prob expectation: {prob_exp}")

## Visualize Training Curves

In [ ]:
# Plot MCal training curves
fig, ax = plot_training_curves(mcal_stats, title='MCal Training Curves')
plt.show()

# Plot Platt training curves  
fig, ax = plot_training_curves(platt_stats, title='Platt Training Curves')
plt.show()

## Compare Results

Compare the calibration performance of different methods.

In [ ]:
# Calculate original expectations
orig_one_hot, orig_prob = get_expectation(ablated_probs)

print("Comparison of Expectations:")
print(f"Original prob expectation: {orig_prob}")
print(f"MCal prob expectation: {get_expectation(mcal_output)[1]}")
print(f"Platt prob expectation: {get_expectation(platt_output)[1]}")
print(f"Temperature prob expectation: {get_expectation(temp_output)[1]}")
print(f"Target (uniform): {torch.ones(d) / d}")

## Conclusion

This demo shows how to use the MCal framework to calibrate probability distributions using different methods:

1. **MCal**: Vector scaling with learnable class-specific parameters
2. **Platt Calibrator**: Platt scaling with logit transformation
3. **Temperature Scaling**: Simple temperature parameter scaling

Each method has different strengths and is suitable for different scenarios.